In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import scanpy as sc
import scvelo as scv
import numpy as np

adata = sc.read_h5ad("./data/pancreas/pancreas_inferred_velocity.h5ad")
n_pcs = 50

# Use the matrix YOU trust (usually adata.X or adata.layers["Ms"])
X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X

# Optional: standardize (recommended if you want PCA to behave)
X = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

# Optional: "stretch PCA" (identity by default; customize if needed)
stretch = np.ones(X.shape[1])   # <- replace with gene-wise stretch if desired
X = X * stretch

# PCA
pca = PCA(n_components=n_pcs, svd_solver="arpack", random_state=0)
X_pca = pca.fit_transform(X)

# --------------------------
# OVERWRITE PCA IN ADATA
# --------------------------
adata.obsm["X_pca"] = X_pca
adata.varm["PCs"] = pca.components_.T
adata.uns["pca"] = {
    "variance": pca.explained_variance_,
    "variance_ratio": pca.explained_variance_ratio_,
}

print("✅ Overwrote PCA:", adata.obsm["X_pca"].shape)

In [ ]:
adata

In [ ]:
scv.tl.velocity(adata, mode="stochastic", vkey="stochastic_velocity")
scv.tl.velocity_graph(adata, vkey="stochastic_velocity")

scv.tl.velocity(adata, mode="dynamical", vkey="dynamical_velocity")
scv.tl.velocity_graph(adata, vkey="dynamical_velocity")

In [ ]:
if "stochastic_velocity_pca" not in adata.obsm:
    print("Computing stochastic_velocity_pca...")
    scv.tl.velocity_embedding(adata, basis="pca", vkey="stochastic_velocity")

V_pca_stoch = adata.obsm["stochastic_velocity_pca"]

# Compute ONLY if missing
if "dynamical_velocity_pca" not in adata.obsm:
    print("Computing dynamical_velocity_pca...")
    scv.tl.velocity_embedding(adata, basis="pca", vkey="dynamical_velocity")

In [ ]:
from scripts.VectorFieldEmbedder import *
from scripts.plotting import *

X = adata.obsm["X_pca"]
V = adata.obsm["stochastic_velocity_pca"]
X_umap = adata.obsm["X_umap"]
cell_type = adata.obs["clusters"]

emb = VectorFieldEmbedder(X, V, dist_method="phase",
                          dof=30, X_emb = X_umap,
                          embed_kwargs={"n_neighbors":30,
                                                    "min_dist":0.5,
                                                    "spread":1.0})
emb.initialize_embedding(seed=0)
# plot_velocity_streamplot(
#     X_2d=emb.X_emb,
#     tps_vf=emb.tps_vf,
#     scatter_color=cell_type,
#     grid_density=1.0, 
#     stream_density=1.2,
#     scatter_size=200,
#     scatter_alpha=0.2,
#     figsize=(8, 8),
#     aspect=1,
#     vmin=0.0,
#     vmax=1.0,
#     cmap="tab10",
#     grid_size=50
# )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from sklearn.neighbors import NearestNeighbors
from scripts.plotting import compute_velocity_on_grid

# ------------------------------------------------------------------
# Helper: trim grid points outside the data manifold
# ------------------------------------------------------------------
def points_inside_mask(X, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X)
    r = np.median(nn.kneighbors(X)[0][:, -1]) * radius_scale
    idx = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(i) > 0 for i in idx])


# ------------------------------------------------------------------
# 1) Embedding + colors
# ------------------------------------------------------------------
X_emb = emb.X_emb

cluster_key = "clusters"
labels = adata.obs[cluster_key].values
categories = adata.obs[cluster_key].cat.categories
colors = np.asarray(adata.uns[f"{cluster_key}_colors"])

# Canonical mapping
colmap = dict(zip(categories, colors))
cell_colors = np.array([colmap[l] for l in labels])


# ------------------------------------------------------------------
# 2) Velocity grid (mass-filtered + boundary trimmed)
# ------------------------------------------------------------------
Xg, _, _ = compute_velocity_on_grid(
    X_emb, grid_size=22, min_mass=0.01
)

keep = points_inside_mask(X_emb, Xg)
Xg = Xg[keep]
Vg = emb.tps_vf.predict(Xg)


# ------------------------------------------------------------------
# 3) Plot
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6))

# Cell scatter
ax.scatter(
    X_emb[:, 0], X_emb[:, 1],
    c=cell_colors, s=60, alpha=0.35, linewidths=0
)

# Velocity field
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy", scale_units="xy", scale=0.1,
    width=0.004,
    headwidth=4.5, headlength=3.0, headaxislength=2.3,
    minlength=0.2,
    color="k", alpha=0.9
)

# ------------------------------------------------------------------
# 4) Cell-type labels (median center + white outline)
# ------------------------------------------------------------------
for ct in np.unique(labels):
    idx = labels == ct
    if idx.sum() == 0:
        continue

    x_center = np.median(X_emb[idx, 0])
    y_center = np.median(X_emb[idx, 1])

    txt = ax.text(
        x_center,
        y_center,
        str(ct),
        ha="center",
        va="center",
        fontsize=10,
        color="black",
        zorder=10,
        alpha=0.7
    )

    txt.set_path_effects([
        pe.Stroke(linewidth=2.5, foreground="white"),
        pe.Normal(),
    ])


# ------------------------------------------------------------------
# 5) Clean axes (scVelo style)
# ------------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values():
    s.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from scripts.FieldReconstructionEvaluator import *

cell_types = adata.obs["clusters"].unique().tolist()

results_by_cell_type = {}

# --- copy once (shared TPS surfaces) ---
emb.X_gene = emb.X
emb.V_gene = emb.V
emb.tps_gene = emb.tps
emb.tps_gene_vf = emb.tps_vf

for ct in cell_types:
    print(f"\n=== Evaluating cell type: {ct} ===")

    # --- cell indices for this type ---
    cell_idx = np.where(adata.obs["clusters"].values == ct)[0]

    if len(cell_idx) < 10:
        print(f"[Skip] Too few cells ({len(cell_idx)})")
        continue

    # --- evaluator ---
    evaluator = FieldReconstructionEvaluator(
        emb,
        cell_idx=cell_idx
    )

    # --- main evaluation ---
    res = evaluator.evaluate_gene_fit()

    # --- (optional) permutation test ---
    # perm = evaluator.permutation_pvals(
    #     res,
    #     n_perm=500,
    #     n_bins=20,
    #     random_state=0,
    # )

    results_by_cell_type[ct] = {
        "n_cells": len(cell_idx),
        "eval": res,
        # "perm": perm,
    }

print("\nDone. Results stored in `results_by_cell_type`.")

In [ ]:
from adjustText import adjust_text
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Pick cell type
# ------------------------------------------------------------
ct = "Ductal"
res = results_by_cell_type[ct]["eval"]

# ------------------------------------------------------------
# PC-level concordance
# Each point corresponds to one principal component
# ------------------------------------------------------------
n_pcs = 50
pcs = np.array([f"PC{i}" for i in range(1, n_pcs + 1)])
expr_corr = np.array(res["expr_corr_gene"])   # PC-wise expression concordance
vel_corr  = np.array(res["vel_corr_gene"])    # PC-wise velocity concordance

# valid points
valid = np.isfinite(expr_corr) & np.isfinite(vel_corr)
pcs = pcs[valid]
expr_corr = expr_corr[valid]
vel_corr  = vel_corr[valid]

# ------------------------------------------------------------
# Highlight PCs 1–10
# ------------------------------------------------------------
top_pcs = {f"PC{i}" for i in range(1, 11)}
is_top = np.array([pc in top_pcs for pc in pcs])

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
plt.figure(figsize=(5, 5))

# background PCs (PC11+)
plt.scatter(
    expr_corr[~is_top],
    vel_corr[~is_top],
    s=90,
    color="grey",
    alpha=0.6,
    edgecolor="none",
    zorder=1
)

# highlighted PCs (PC1–10)
plt.scatter(
    expr_corr[is_top],
    vel_corr[is_top],
    s=260,
    color="steelblue",
    alpha=0.7,
    lw=0.9,
    zorder=3
)

# ------------------------------------------------------------
# Text labels for top PCs
# ------------------------------------------------------------
# texts = []
# for x, y, pc in zip(expr_corr[is_top], vel_corr[is_top], pcs[is_top]):
#     texts.append(
#         plt.text(
#             x, y, pc,
#             fontsize=13,
#             weight="bold",
#             ha="left",
#             va="bottom"
#         )
#     )

# adjust_text(
#     texts,
#     x=expr_corr[is_top],
#     y=vel_corr[is_top],
#     arrowprops=dict(arrowstyle="-", lw=1.1, color="black"),
#     force_text=1.4,
#     force_points=1.2,
#     expand_text=(1.4, 1.4),
#     expand_points=(1.3, 1.3),
# )

# ------------------------------------------------------------
# Axes & styling
# ------------------------------------------------------------
plt.xlim(0, 1.1)
plt.ylim(0, 1.1)

ticks = np.linspace(0, 1, 6)
plt.xticks(ticks, [f"{t:.1f}" for t in ticks], fontsize=12)
plt.yticks(ticks, [f"{t:.1f}" for t in ticks], fontsize=12)

plt.grid(True, which="major", linestyle="--", alpha=0.4)

plt.xlabel(r"Expression concordance $(r)$", fontsize=16)
plt.ylabel(r"Velocity concordance $(r)$", fontsize=16)
plt.title(f"{ct} cells", fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
import math
import os

# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------
cell_types = list(results_by_cell_type.keys())
n_ct = len(cell_types)

n_pcs = 50
pcs_all = np.array([f"PC{i}" for i in range(1, n_pcs + 1)])
top_pcs = {f"PC{i}" for i in range(1, 11)}

# grid layout
ncols = 3
nrows = math.ceil(n_ct / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(5 * ncols, 5 * nrows),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

# ------------------------------------------------------------
# Loop over cell types
# ------------------------------------------------------------
for ax, ct in zip(axes, cell_types):
    res = results_by_cell_type[ct]["eval"]

    expr_corr = np.array(res["expr_corr_gene"])
    vel_corr  = np.array(res["vel_corr_gene"])

    valid = np.isfinite(expr_corr) & np.isfinite(vel_corr)
    pcs = pcs_all[valid]
    expr_corr = expr_corr[valid]
    vel_corr  = vel_corr[valid]

    is_top = np.array([pc in top_pcs for pc in pcs])

    # background PCs
    ax.scatter(
        expr_corr[~is_top],
        vel_corr[~is_top],
        s=70,
        color="grey",
        alpha=0.6,
        edgecolor="none",
        zorder=1
    )

    # highlighted PCs
    ax.scatter(
        expr_corr[is_top],
        vel_corr[is_top],
        s=180,
        color="steelblue",
        alpha=0.7,
        lw=0.8,
        zorder=3
    )

    # labels for top PCs
    texts = []
    for x, y, pc in zip(expr_corr[is_top], vel_corr[is_top], pcs[is_top]):
        texts.append(
            ax.text(
                x, y, pc,
                fontsize=11,
                weight="bold",
                ha="left",
                va="bottom"
            )
        )

    adjust_text(
        texts,
        ax=ax,
        arrowprops=dict(arrowstyle="-", lw=0.9, color="black"),
        force_text=1.2,
        force_points=1.0,
        expand_text=(1.2, 1.2),
        expand_points=(1.2, 1.2),
    )

    ax.set_title(ct, fontsize=16)
    ax.set_xlim(0, 1.1)
    ax.set_ylim(0, 1.1)
    ax.grid(True, linestyle="--", alpha=0.4)

# ------------------------------------------------------------
# Remove empty panels
# ------------------------------------------------------------
for ax in axes[len(cell_types):]:
    ax.axis("off")

# ------------------------------------------------------------
# Shared labels
# ------------------------------------------------------------
fig.text(
    0.5, 0.04,
    r"Expression concordance $(r)$",
    ha="center",
    fontsize=18
)
fig.text(
    0.04, 0.5,
    r"Velocity concordance $(r)$",
    va="center",
    rotation="vertical",
    fontsize=18
)

plt.tight_layout(rect=[0.06, 0.06, 1, 1])

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
outdir = "./figures/pancreas"
os.makedirs(outdir, exist_ok=True)

plt.savefig(
    os.path.join(outdir, "pc_concordance_by_cell_type.png"),
    dpi=200
)
plt.close()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ------------------------------------------------------------------
# Context: PCA space
# ------------------------------------------------------------------
X_pca = adata.obsm["X_pca"]
V_pca = adata.obsm["stochastic_velocity_pca"]

n_pcs = 10   # show PC1–PC10 in the heatmap
pcs = [f"PC{i}" for i in range(1, n_pcs + 1)]
cell_types = list(results_by_cell_type.keys())

# --- colormap: white -> red (R^2 style) ----------------------------
cmap_r2 = LinearSegmentedColormap.from_list(
    "white_red",
    ["#ffffff", "#fddede", "#f46d6d", "#b30000"]
)

# ------------------------------------------------------------------
# Collect PC-level reconstruction R^2
# ------------------------------------------------------------------
expr_pca_rows = []
vel_pca_rows  = []

for ct in cell_types:
    res = results_by_cell_type[ct]["eval"]

    expr_corr = np.asarray(res["expr_corr_gene"])  # length >= n_pcs
    vel_corr  = np.asarray(res["vel_corr_gene"])

    row_expr = {f"PC{i+1}": expr_corr[i] for i in range(n_pcs)}
    row_vel  = {f"PC{i+1}": vel_corr[i]  for i in range(n_pcs)}

    expr_pca_rows.append(pd.Series(row_expr, name=ct))
    vel_pca_rows.append(pd.Series(row_vel,  name=ct))

expr_pca_df = pd.DataFrame(expr_pca_rows)
vel_pca_df  = pd.DataFrame(vel_pca_rows)

# ------------------------------------------------------------------
# Cell-type ordering (paper order)
# ------------------------------------------------------------------
cell_type_order = [
    "Ductal",
    "Ngn3 low EP",
    "Ngn3 high EP",
    "Pre-endocrine",
    "Alpha",
    "Beta",
    "Delta",
    "Epsilon",
]

expr_pca_df = expr_pca_df.reindex(cell_type_order)
vel_pca_df  = vel_pca_df.reindex(cell_type_order)

# ------------------------------------------------------------------
# Annotation formatting (no leading zero)
# ------------------------------------------------------------------
def fmt_no_leading_zero(x):
    if np.isnan(x):
        return ""
    s = f"{x:.2f}"
    return s[1:] if s.startswith("0") else s

expr_pca_annot = expr_pca_df.applymap(fmt_no_leading_zero)
vel_pca_annot  = vel_pca_df.applymap(fmt_no_leading_zero)

# ------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------
fig, axes = plt.subplots(
    1, 2,
    figsize=(1.2 * len(pcs) + 6, 0.55 * len(cell_type_order)),
    sharey=True
)

sns.heatmap(
    expr_pca_df,
    ax=axes[0],
    cmap=cmap_r2,
    vmin=0.4, vmax=1,
    linewidths=0.4,
    cbar=False,
    annot=expr_pca_annot,
    fmt="",
    annot_kws={"fontsize": 14}
)

sns.heatmap(
    vel_pca_df,
    ax=axes[1],
    cmap=cmap_r2,
    vmin=0.4, vmax=1,
    linewidths=0.4,
    cbar=False,
    annot=vel_pca_annot,
    fmt="",
    annot_kws={"fontsize": 14}
)

# ------------------------------------------------------------------
# Shared colorbar
# ------------------------------------------------------------------
cbar_ax = fig.add_axes([0.92, 0.25, 0.02, 0.5])

sm = plt.cm.ScalarMappable(
    cmap=cmap_r2,
    norm=plt.Normalize(vmin=0.4, vmax=1)
)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_ticks([0.4, 0.7, 1.0])
cbar.set_ticklabels(["0.4", "0.7", "1.0"])
cbar.ax.tick_params(labelsize=18)

# ------------------------------------------------------------------
# Axes cosmetics
# ------------------------------------------------------------------
for ax in axes:
    ax.tick_params(axis="x", labelrotation=45, labelsize=18)
    ax.tick_params(axis="y", labelsize=18)

axes[0].set_title("Expression", fontsize=30)
axes[1].set_title("Velocity", fontsize=30)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

def smooth_vector_field_knn(X, V, k=30):
    """
    Locally smooth vector field by averaging k-nearest-neighbor velocities.
    """
    nbrs = NearestNeighbors(n_neighbors=k, algorithm="auto").fit(X)
    _, idx = nbrs.kneighbors(X)

    V_smooth = np.zeros_like(V)
    for i in range(X.shape[0]):
        V_smooth[i] = V[idx[i]].mean(axis=0)

    return V_smooth

def plot_2d_quiver(
    X,
    vectors,
    color=None,
    rgba_colors=None,          # NEW
    s=10,
    alpha=0.3,
    scale=1,
    cmap="coolwarm",
    use_normalized=False,
    title="",
    show_legend=True,
    show_colorbar=True,
    legend_labels=None,
    legend_font_size=12,
    legend_pos="upper left",
    max_categories=6,
):
    """
    Plots a 2D quiver with flexible color handling:
    - explicit RGBA colors -> used directly
    - categorical numeric colors -> legend
    - continuous numeric colors  -> colorbar
    """

    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    from matplotlib import cm
    from matplotlib.lines import Line2D

    # ------------------------------------------------------------
    # Normalize vectors if requested
    # ------------------------------------------------------------
    if use_normalized:
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms[norms == 0] = 1
        vectors = vectors / norms

    # ------------------------------------------------------------
    # Color handling
    # ------------------------------------------------------------
    if rgba_colors is not None:
        rgba_colors = np.asarray(rgba_colors)
        is_categorical = True
        cmap_obj = None
        norm = None
    else:
        color = np.asarray(color)
        unique_vals = np.unique(color[~np.isnan(color)])
        is_categorical = len(unique_vals) <= max_categories

        cmap_obj = cm.get_cmap(cmap)

        if is_categorical:
            norm = mcolors.Normalize(vmin=unique_vals.min(), vmax=unique_vals.max())
        else:
            norm = mcolors.Normalize(vmin=np.nanmin(color), vmax=np.nanmax(color))

        rgba_colors = cmap_obj(norm(color))

    # ------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="white")

    ax.scatter(
        X[:, 0],
        X[:, 1],
        color=rgba_colors,
        s=s,
        alpha=alpha,
        edgecolors="none",
    )

    ax.quiver(
        X[:, 0],
        X[:, 1],
        vectors[:, 0],
        vectors[:, 1],
        color=rgba_colors,
        angles="xy",
        scale_units="xy",
        scale=scale,
        width=0.004,
        alpha=0.8,
    )

    # ------------------------------------------------------------
    # Legend for categorical
    # ------------------------------------------------------------
    if is_categorical and show_legend:
        handles = []

        if rgba_colors is not None:
            # explicit color legend
            unique_labels = np.unique(legend_labels)
            for label in unique_labels:
                handles.append(
                    Line2D(
                        [0], [0],
                        marker="o",
                        linestyle="",
                        markerfacecolor=colmap[label],
                        markeredgecolor="none",
                        markersize=8,
                        label=label,
                    )
                )
        else:
            if legend_labels is None:
                legend_labels = [str(v) for v in unique_vals]

            for v, label in zip(unique_vals, legend_labels):
                handles.append(
                    Line2D(
                        [0], [0],
                        marker="o",
                        linestyle="",
                        markerfacecolor=cmap_obj(norm(v)),
                        markeredgecolor="none",
                        markersize=8,
                        label=label,
                    )
                )

        ax.legend(
            handles=handles,
            loc=legend_pos,
            frameon=False,
            fontsize=legend_font_size,
        )

    # ------------------------------------------------------------
    # Colorbar for continuous
    # ------------------------------------------------------------
    if (rgba_colors is None) and (not is_categorical) and show_colorbar:
        sm = cm.ScalarMappable(norm=norm, cmap=cmap_obj)
        sm.set_array([])
        cbar = plt.colorbar(
            sm,
            ax=ax,
            pad=0.01,
            fraction=0.035,
            shrink=0.75
        )
        cbar.ax.tick_params(labelsize=9)

    # ------------------------------------------------------------
    # Styling
    # ------------------------------------------------------------
    ax.set_title(title, fontsize=12)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

    plt.show()


cluster_key = "clusters"
labels = adata.obs[cluster_key].values
categories = adata.obs[cluster_key].cat.categories
colors = np.asarray(adata.uns[f"{cluster_key}_colors"])

# Canonical mapping
colmap = dict(zip(categories, colors))


pc_x = 2
pc_y = 7

X_pca = adata.obsm["X_pca"][:, [pc_x, pc_y]]
V_pca = adata.obsm["stochastic_velocity_pca"][:, [pc_x, pc_y]]

cell_types = adata.obs["clusters"].values

mask = np.isin(cell_types, ["Ductal", "Ngn3 low EP"])

X_sel = X_pca[mask]
V_sel = V_pca[mask]
ct_sel = cell_types[mask]
V_smooth = smooth_vector_field_knn(
    X_sel,
    V_sel,
    k=30,   # feel free to try 20–50
)

color = np.zeros(len(ct_sel))
color[ct_sel == "Ngn3 low EP"] = 1.0
rgba_colors = np.array([colmap[l] for l in ct_sel])

plot_2d_quiver(
    X_sel,
    V_sel,
    rgba_colors=rgba_colors,   # <-- key change
    s=40,
    alpha=0.5,
    scale=0.4,
    legend_labels=np.unique(ct_sel),
    legend_pos="upper left",
)

# color = adata.obs["S_score"] - adata.obs["G2M_score"]
# color = color[mask]

# plot_2d_quiver(
#     X_sel,
#     V_smooth,
#     color=color,
#     s=30,
#     alpha=0.2,
#     scale=0.4,
#     cmap="viridis",
# )


In [ ]:
pc_x = 2
pc_y = 5

X_pca = adata.obsm["X_pca"][:, [pc_x, pc_y]]
V_pca = adata.obsm["stochastic_velocity_pca"][:, [pc_x, pc_y]]

cell_types = adata.obs["clusters"].values

mask = np.isin(cell_types, ["Ductal", "Ngn3 low EP"])

X_sel = X_pca[mask]
V_sel = V_pca[mask]
ct_sel = cell_types[mask]
V_smooth = smooth_vector_field_knn(
    X_sel,
    V_sel,
    k=30,   # feel free to try 20–50
)

color = np.zeros(len(ct_sel))
color[ct_sel == "Ngn3 low EP"] = 1.0
rgba_colors = np.array([colmap[l] for l in ct_sel])

plot_2d_quiver(
    X_sel,
    V_sel,
    rgba_colors=rgba_colors,   # <-- key change
    s=40,
    alpha=0.5,
    scale=0.4,
    legend_labels=np.unique(ct_sel),
    legend_pos="upper left",
)

color = adata.obs["S_score"] - adata.obs["G2M_score"]
color = color[mask]

plot_2d_quiver(
    X_sel,
    V_smooth,
    color=color,
    s=30,
    alpha=0.2,
    scale=0.4,
    cmap="viridis",
)

In [ ]:
pc_x = 3
pc_y = 1

X_pca = adata.obsm["X_pca"][:, [pc_x, pc_y]]
V_pca = adata.obsm["stochastic_velocity_pca"][:, [pc_x, pc_y]]

cell_types = adata.obs["clusters"].values

mask = np.isin(cell_types, ["Alpha", "Beta", "Delta", "Epsilon"])

X_sel = X_pca[mask]
V_sel = V_pca[mask]
ct_sel = cell_types[mask]
V_smooth = smooth_vector_field_knn(
    X_sel,
    V_sel,
    k=30,   # feel free to try 20–50
)

type_to_int = {
    "Alpha": 0,
    "Beta": 1,
    "Delta": 2,
    "Epsilon": 3,
}
color = np.array([type_to_int[t] for t in ct_sel])
rgba_colors = np.array([colmap[l] for l in ct_sel])

plot_2d_quiver(
    X_sel,
    V_sel,
    rgba_colors=rgba_colors,   # <-- key change
    s=40,
    alpha=0.5,
    scale=0.4,
    legend_font_size=14,
    legend_labels=np.unique(ct_sel),
    legend_pos="upper right",
)

In [ ]:
# ------------------------------------------------------------
# Select PCs (PC1 vs PC2 → indices 0, 1)
# ------------------------------------------------------------
pc_x = 0   # PC1
pc_y = 1   # PC2

X_pca = adata.obsm["X_pca"][:, [pc_x, pc_y]]
V_pca = adata.obsm["stochastic_velocity_pca"][:, [pc_x, pc_y]]

# ------------------------------------------------------------
# Select cell types
# ------------------------------------------------------------
cell_types = adata.obs["clusters"].values
mask = np.isin(cell_types, ["Pre-endocrine", "Ngn3 high EP"])

X_sel = X_pca[mask]
V_sel = V_pca[mask]
ct_sel = cell_types[mask]

# ------------------------------------------------------------
# Smooth velocity field
# ------------------------------------------------------------
V_smooth = smooth_vector_field_knn(
    X_sel,
    V_sel,
    k=30
)

# ------------------------------------------------------------
# Categorical coloring
# ------------------------------------------------------------
type_to_int = {
    "Pre-endocrine": 0,
    "Ngn3 high EP": 1,
}
color = np.array([type_to_int[t] for t in ct_sel])
rgba_colors = np.array([colmap[l] for l in ct_sel])

plot_2d_quiver(
    X_sel,
    V_sel,
    rgba_colors=rgba_colors,   # <-- key change
    s=40,
    alpha=0.4,
    scale=0.4,
    legend_font_size=16,
    legend_labels=np.unique(ct_sel),
    legend_pos="lower left",
)

In [ ]:
pc_x = 4
pc_y = 5

X_pca = adata.obsm["X_pca"][:, [pc_x, pc_y]]
V_pca = adata.obsm["stochastic_velocity_pca"][:, [pc_x, pc_y]]

cell_types = adata.obs["clusters"].values

mask = np.isin(cell_types, ["Beta", "Epsilon"])

X_sel = X_pca[mask]
V_sel = V_pca[mask]
ct_sel = cell_types[mask]
V_smooth = smooth_vector_field_knn(
    X_sel,
    V_sel,
    k=30,   # feel free to try 20–50
)

type_to_int = {
    "Alpha": 0,
    "Beta": 1,
    "Delta": 2,
    "Epsilon": 3,
}
color = np.array([type_to_int[t] for t in ct_sel])
rgba_colors = np.array([colmap[l] for l in ct_sel])

plot_2d_quiver(
    X_sel,
    V_sel,
    rgba_colors=rgba_colors,   # <-- key change
    s=40,
    alpha=0.5,
    scale=0.1,
    legend_font_size=14,
    legend_labels=np.unique(ct_sel),
    legend_pos="upper right",
)

In [ ]:
pc_x = 3
pc_y = 1

X_pca = adata.obsm["X_pca"][:, [pc_x, pc_y]]
V_pca = adata.obsm["stochastic_velocity_pca"][:, [pc_x, pc_y]]

cell_types = adata.obs["clusters"].values

mask = np.isin(cell_types, ["Beta", "Epsilon"])

X_sel = X_pca[mask]
V_sel = V_pca[mask]
ct_sel = cell_types[mask]
V_smooth = smooth_vector_field_knn(
    X_sel,
    V_sel,
    k=30,   # feel free to try 20–50
)

type_to_int = {
    "Alpha": 0,
    "Beta": 1,
    "Delta": 2,
    "Epsilon": 3,
}
color = np.array([type_to_int[t] for t in ct_sel])
rgba_colors = np.array([colmap[l] for l in ct_sel])

plot_2d_quiver(
    X_sel,
    V_sel,
    rgba_colors=rgba_colors,   # <-- key change
    s=40,
    alpha=0.7,
    scale=0.3,
    legend_font_size=14,
    legend_labels=np.unique(ct_sel),
    legend_pos="upper right",
)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gseapy as gp

# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------
outdir = "./figures/pancreas"
os.makedirs(outdir, exist_ok=True)

genes = adata.var_names.to_numpy()
PCs = adata.varm["PCs"]
n_pcs = min(10, PCs.shape[1])

gene_sets = "GO_Biological_Process_2021"
organism = "Mouse"

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------
def prep_gsea_panel(df, n_terms=8):
    df = df.copy()
    df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
    df["FDR q-val"] = pd.to_numeric(df["FDR q-val"], errors="coerce")

    if "Term" in df.columns:
        df["term"] = df["Term"].astype(str)
    else:
        df["term"] = df.index.astype(str)

    df = df[df["FDR q-val"] < 0.05]
    if df.empty:
        return None

    df = (
        df.sort_values("NES", ascending=False)
          .head(n_terms)
          .sort_values("NES")
    )
    return df


# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, axes = plt.subplots(
    nrows=5, ncols=2,
    figsize=(30, 25),
    sharex=False
)

axes = axes.flatten()

for pc_idx in range(n_pcs):
    ax = axes[pc_idx]
    pc_label = f"PC{pc_idx + 1}"

    print(f"[GSEA] {pc_label}")

    # ranked list
    rnk = pd.DataFrame({
        "gene": genes,
        "loading": PCs[:, pc_idx]
    })

    res = gp.prerank(
        rnk=rnk,
        gene_sets=gene_sets,
        organism=organism,
        permutation_num=1000,
        min_size=15,
        max_size=500,
        outdir=None,
        seed=0,
        verbose=False
    )

    df_plot = prep_gsea_panel(res.res2d)

    if df_plot is None:
        # empty panel with text
        ax.text(
            0.5, 0.5,
            "No significant terms",
            ha="center", va="center",
            fontsize=36,
            color="gray",
            transform=ax.transAxes
        )
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)
        ax.set_title(pc_label)
        continue

    # plot
    ax.barh(df_plot["term"], df_plot["NES"], color="black")
    ax.set_title(pc_label)
    ax.set_xlabel("NES")

plt.tight_layout()
plt.savefig(
    os.path.join(outdir, "pc1_10_gsea_grid.png"),
    dpi=200
)
plt.close()

In [ ]:
from scripts.pseudotime import StochasticPseudotime

spt = StochasticPseudotime(
    X=emb.X_emb,              # (n_cells, d)
    vector_field=emb.tps_vf,  # must have .predict()
    tps=emb.tps               # optional but recommended
)

root = spt.find_root(
    n_simulations_per_cell=5,
    max_steps=100
)

tau = spt.compute_pseudotime(
    root
)

In [ ]:
import numpy as np
from scripts.FieldReconstructionEvaluator import *

# --- pseudotime (normalized) ---
pt = adata.obs["velocity_pseudotime"].values
pt = (pt - pt.min()) / (pt.max() - pt.min())

# --- sliding window parameters (pseudotime units) ---
pt_window = 0.1
pt_step   = 0.02
n_pcs     = 10
min_cells = 40

results_by_window = {}

# --- traces to plot ---
expr_corr_traces = []   # (n_windows, n_pcs)
vel_corr_traces  = []   # (n_windows, n_pcs)
pt_centers       = []

win_id = 0
pt_min, pt_max = pt.min(), pt.max()

t = pt_min
while t + pt_window <= pt_max:
    # --- select cells in pseudotime window ---
    cell_idx = np.where((pt >= t) & (pt < t + pt_window))[0]

    if len(cell_idx) < min_cells:
        t += pt_step
        continue

    evaluator = FieldReconstructionEvaluator(
        emb,
        cell_idx=cell_idx
    )

    res = evaluator.evaluate_gene_fit()

    # --- extract correlations (PC-wise) ---
    expr_corr = res["expr_corr_gene"][:n_pcs]
    vel_corr  = res["vel_corr_gene"][:n_pcs]

    pt_center = np.median(pt[cell_idx])

    results_by_window[win_id] = {
        "cell_idx": cell_idx,
        "pt_center": pt_center,
        "pt_range": (t, t + pt_window),
        "n_cells": len(cell_idx),
        "expr_corr_gene": expr_corr,
        "vel_corr_gene": vel_corr,
        "eval": res,
    }

    expr_corr_traces.append(expr_corr)
    vel_corr_traces.append(vel_corr)
    pt_centers.append(pt_center)

    win_id += 1
    t += pt_step

# --- stack for plotting ---
expr_corr_traces = np.vstack(expr_corr_traces)   # (n_windows, n_pcs)
vel_corr_traces  = np.vstack(vel_corr_traces)    # (n_windows, n_pcs)
pt_centers       = np.asarray(pt_centers)

In [ ]:
from scipy.ndimage import gaussian_filter1d

# sigma in units of windows (not pseudotime)
sigma = 1.5

expr_corr_smooth = gaussian_filter1d(expr_corr_traces, sigma=sigma, axis=0)
vel_corr_smooth  = gaussian_filter1d(vel_corr_traces,  sigma=sigma, axis=0)


# --- PC1–PC6 visual encoding ---
colors = [
    "#7f0000",  # PC1 - dark red
    "#a50f15",  # PC2
    "#cb181d",  # PC3
    "#ef3b2c",  # PC4
    "#fb6a4a",  # PC5
    "#fcae91",  # PC6 - light red
]

line_widths = [3.2, 2.6, 2.1, 1.6, 1.2, 0.9]
alphas      = [0.95, 0.9, 0.8, 0.7, 0.55, 0.45]

n_show = 6

plt.figure(figsize=(12, 3.2))

for i in range(n_show):
    plt.plot(
        pt_centers,
        expr_corr_smooth[:, i],
        color=colors[i],
        linewidth=line_widths[i],
        alpha=alphas[i],
        label=f"PC{i+1}"
    )

plt.xlabel("Velocity pseudotime")
plt.ylabel("Expression correlation (r)")
plt.title("Gene expression alignment along pseudotime (PC1–PC6)")

plt.legend(
    frameon=False,
    fontsize=9,
    ncol=3,
    loc="lower left"
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 3.2))

for i in range(n_show):
    plt.plot(
        pt_centers,
        vel_corr_smooth[:, i],
        color=colors[i],
        linewidth=line_widths[i],
        alpha=alphas[i],
        label=f"PC{i+1}"
    )

plt.xlabel("Velocity pseudotime")
plt.ylabel("Velocity correlation (r)")
plt.title("Velocity field alignment along pseudotime (PC1–PC6)")

plt.legend(
    frameon=False,
    fontsize=9,
    ncol=3,
    loc="lower left"
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- PCs to show ---
n_show = 6

# --- red → light red palette ---
colors = [
    "#7f0000",  # PC1
    "#a50f15",  # PC2
    "#cb181d",  # PC3
    "#ef3b2c",  # PC4
    "#fb6a4a",  # PC5
    "#fcae91",  # PC6
]

line_widths = [3.2, 2.6, 2.1, 1.6, 1.2, 0.9]
alphas      = [0.95, 0.9, 0.8, 0.7, 0.55, 0.45]

# --- figure ---
fig, axes = plt.subplots(
    2, 1,
    figsize=(10.0, 4.0),
    sharex=True,
    gridspec_kw=dict(hspace=0.3)
)

# =========================
# Top: expression r
# =========================
for i in range(n_show):
    axes[0].plot(
        pt_centers,
        expr_corr_traces[:, i],
        color=colors[i],
        linewidth=line_widths[i],
        alpha=alphas[i],
        label=f"PC{i+1}"
    )

axes[0].set_ylim(-0.5, 1.05)
axes[0].set_yticks([-0.5, 0.0, 0.5, 1.0])
axes[0].set_title(
    r"Expression $r(y, \hat{y})$ over pseudotime",
    fontsize=18,
    pad=6
)

# =========================
# Bottom: velocity r
# =========================
for i in range(n_show):
    axes[1].plot(
        pt_centers,
        vel_corr_traces[:, i],
        color=colors[i],
        linewidth=line_widths[i],
        alpha=alphas[i],
        label=f"PC{i+1}"
    )

axes[1].set_ylim(-0.5, 1.05)
axes[1].set_yticks([-0.5, 0.0, 0.5, 1.0])
axes[1].tick_params(axis="x", labelsize=14)

axes[1].legend(
    frameon=False,
    fontsize=12,
    loc="lower left",
    ncol=3
)
axes[1].set_title(
    r"Velocity $r(v, \hat{v})$ over pseudotime",
    fontsize=18,
    pad=6
)

# --- clean spines ---
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

X_umap = adata.obsm["X_umap"]
pt = adata.obs["velocity_pseudotime"].values

fig, ax = plt.subplots(figsize=(5.5, 5.0))

sc = ax.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=pt,
    cmap="viridis",
    s=40,           # larger points
    linewidths=0
)

ax.set_xticks([])
ax.set_yticks([])
ax.set_frame_on(False)

# --- large horizontal colorbar ---
cbar = fig.colorbar(
    sc,
    ax=ax,
    orientation="horizontal",
    fraction=0.10,   # thicker bar
    pad=0.12
)

cbar.set_label("Pseudotime", fontsize=24, labelpad=8)
cbar.set_ticks([0.0, 0.5, 1.0])
cbar.ax.tick_params(labelsize=20)

plt.tight_layout()
plt.show()